In [1]:
import duckdb
import pandas as pd 
PARQUET_PATH = "C:/Users/wertu/Downloads/polymarket_orderbook_2026-02-28T19.parquet"  
con = duckdb.connect()
df = con.execute(f"""
    SELECT data
    FROM read_parquet('{PARQUET_PATH}')
    LIMIT 5
""").fetch_df()
print(df.iloc[0]["data"])

{"update_type": "price_change", "market_id": "0x571db51db13c8e4a409bd85386effa28ca60061105e841913b44822fab30d675", "token_id": "36795769776608273694892366525345913933732415126777409670318872995829106419960", "side": "NO", "best_bid": "0.707", "best_ask": "0.871", "timestamp": 1772305215.3318043, "change_price": "0.174", "change_size": "0", "change_side": "BUY"}


In [2]:
query = f"""
WITH parsed AS (
    SELECT
        timestamp_received AS ts,
        CAST(data AS JSON) AS j
    FROM read_parquet('{PARQUET_PATH}')
)

SELECT
    ts,
    j ->> 'market_id' AS market_id,
    j ->> 'token_id'  AS token_id,
    (j ->> 'best_bid')::DOUBLE AS best_bid,
    (j ->> 'best_ask')::DOUBLE AS best_ask
FROM parsed
WHERE j ->> 'best_ask' IS NOT NULL
"""

df = con.execute(query).fetch_df()

print(df.count())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ts           36849450
market_id    36849450
token_id     36849450
best_bid     36842672
best_ask     36849450
dtype: int64


In [4]:
df["token_id"] = df["token_id"].astype(str).str.replace('"', '')
df["time"] = pd.to_datetime(df["ts"], unit='s')
df = df.sort_values("time")
#Считаем ликвидность каждого рынка
market_liquidity = df.groupby('market_id').size().sort_values(ascending=False)
top_markets = market_liquidity.head(6).index.tolist()
print("Топ-6 рынков по активности:", top_markets)

#симуляция стратегии 
def simulate_negrisk(market_df, threshold=1.0):
    market_df = market_df.sort_values("time")
    pivot = market_df.pivot_table(
        index="time",
        columns="token_id",
        values="best_ask"
    ).sort_index().ffill().dropna()

    signals = []
    open_signal = None

    for ts, row in pivot.iterrows():
        total_cost = row.sum()
        if total_cost < threshold:
            if open_signal is None:
                open_signal = {'ts_start': ts, 'entry_cost': total_cost}
        else:
            if open_signal is not None:
                duration = ts - open_signal['ts_start']
                pnl = 1 - open_signal['entry_cost']
                signals.append({
                    'ts_start': open_signal['ts_start'],
                    'ts_end': ts,
                    'duration_sec': duration.total_seconds(),
                    'entry_cost': open_signal['entry_cost'],
                    'pnl': pnl
                })
                open_signal = None

    
    if open_signal is not None:
        duration = pivot.index.max() - open_signal['ts_start']
        pnl = 1 - open_signal['entry_cost']
        signals.append({
            'ts_start': open_signal['ts_start'],
            'ts_end': pivot.index.max(),
            'duration_sec': duration.total_seconds(),
            'entry_cost': open_signal['entry_cost'],
            'pnl': pnl
        })

    df_signals = pd.DataFrame(signals)
    if df_signals.empty:
        stats = {
            'total_signals': 0,
            'total_pnl': 0,
            'avg_pnl': 0,
            'avg_duration_sec': 0,
            'max_duration_sec': 0
        }
    else:
        stats = {
            'total_signals': len(df_signals),
            'total_pnl': df_signals['pnl'].sum(),
            'avg_pnl': df_signals['pnl'].mean(),
            'avg_duration_sec': df_signals['duration_sec'].mean(),
            'max_duration_sec': df_signals['duration_sec'].max()
        }
  
    return df_signals, stats

#Симуляция по каждому из рынков
result = []
for mid in top_markets:
    liq = market_liquidity.loc[mid]
    print(f" Симуляция для рынка {mid} (liquidity: {liq:,} rows)")
    market_df = df[df['market_id'] == mid]
    df_signals, stats = simulate_negrisk(market_df, threshold=1.0)
    result.append({
        "market_id": mid,
        "liquidity": market_liquidity.loc[mid],
        "total_signals": stats["total_signals"],
        "total_pnl": stats["total_pnl"],
        "avg_pnl": stats["avg_pnl"],
        "avg_duration_sec": stats["avg_duration_sec"],
        "max_duration_sec": stats["max_duration_sec"]
    })
    print(stats)

Топ-6 рынков по активности: ['0x0a96f856b77e5f1cd09b1fbf9c1873cd1c05f6e90523e3d4b5d4e3f1b08ce4b7', '0x3b296112d3b473b2c142548c51be0ea328ae3b16f84b5797c5a28ed683c46d8a', '0x449b80a1dea4b32323ef7751350e84efe7986683407a387019233214fd2440b2', '0xf6e40592172ce387aed742c2bb2e77ee74715b6f6f63b34fb4681a4501b54ee5', '0xca5c2396691263fab599c30b99e2dd105aaa237482b9931ac1dea4b51f02b208', '0xa1e43a84fcd0582e18e0fcdef3ce1a06539c6a073570ab294240940756103666']
 Симуляция для рынка 0x0a96f856b77e5f1cd09b1fbf9c1873cd1c05f6e90523e3d4b5d4e3f1b08ce4b7 (liquidity: 439,370 rows)
{'total_signals': 1, 'total_pnl': np.float64(0.06999999999999995), 'avg_pnl': np.float64(0.06999999999999995), 'avg_duration_sec': np.float64(0.004), 'max_duration_sec': 0.004}
 Симуляция для рынка 0x3b296112d3b473b2c142548c51be0ea328ae3b16f84b5797c5a28ed683c46d8a (liquidity: 408,563 rows)
{'total_signals': 2, 'total_pnl': np.float64(0.050000000000000044), 'avg_pnl': np.float64(0.025000000000000022), 'avg_duration_sec': np.float64(0.

In [5]:
df_results = pd.DataFrame(result)
df_results_clean = df_results.copy()
df_results_clean["liquidity"] = df_results_clean["liquidity"].astype(int)
df_results_clean["total_pnl"] = df_results_clean["total_pnl"].astype(float).round(4)
df_results_clean["avg_pnl"] = df_results_clean["avg_pnl"].astype(float).round(4)
df_results_clean["avg_duration_sec"] = df_results_clean["avg_duration_sec"].astype(float).round(6)
df_results_clean["max_duration_sec"] = df_results_clean["max_duration_sec"].astype(float).round(6)
df_results_clean = df_results_clean.rename(columns={
    "market_id": "Market",
    "liquidity": "Liquidity (events)",
    "total_signals": "Signals",
    "total_pnl": "Total PnL ($)",
    "avg_pnl": "Avg PnL ($)",
    "avg_duration_sec": "Avg Duration (s)",
    "max_duration_sec": "Max Duration (s)"
})
df_results_clean

,Market,Liquidity (events),Signals,Total PnL ($),Avg PnL ($),Avg Duration (s),Max Duration (s)
0,0x0a96f856b77e5f1cd09b1fbf9c1873cd1c05f6e90523...,439370,1,0.07,0.0700,0.004000,0.004
1,0x3b296112d3b473b2c142548c51be0ea328ae3b16f84b...,408563,2,0.05,0.0250,0.006000,0.007
2,0x449b80a1dea4b32323ef7751350e84efe7986683407a...,338455,0,0.00,0.0000,0.000000,0.000
3,0xf6e40592172ce387aed742c2bb2e77ee74715b6f6f63...,293947,3,0.04,0.0133,0.002667,0.003
4,0xca5c2396691263fab599c30b99e2dd105aaa237482b9...,247901,0,0.00,0.0000,0.000000,0.000
5,0xa1e43a84fcd0582e18e0fcdef3ce1a06539c6a073570...,208238,0,0.00,0.0000,0.000000,0.000
